# Import Primary Libraries and Functions

In [2]:
import pandas as pd
import numpy as np
import joblib
import sys
from pathlib import Path

# Import functions from src/features.py
sys.path.append(
    str(Path().resolve().parent)
)
from src.features import (
    team_rating,
    add_ratings,
    assign_tournament_weight,
    add_tournament_weight,
    add_host_advantage,
    add_elo_features,
    add_recent_stats,
    create_match_features
)

/Users/edenaong/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/edenaong/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Load Data and Model

In [3]:
fixtures = pd.read_csv("../data/processed/wc2026_matches.csv")
elo_ratings = pd.read_csv("../data/processed/elo_clean.csv")
players = pd.read_csv("../data/processed/players_clean.csv")
matches = pd.read_csv("../data/processed/matches.csv")
wc2026_draw = pd.read_csv("../data/processed/wc2026_draw_clean.csv")
mapping = pd.read_csv("../data/raw/round_of_32.csv")

model = joblib.load("best_model.pkl")

In [3]:
# Renaming country column to host_country for clarity
fixtures.rename(columns={"country": "host_country"}, inplace=True)

# Build Features

## Squad Ratings
The squad ratings are derived from FC26. To prevent data leakage, they were not used as features in model training. Thus, they will not directly be used as features during simulation as well. 

In [4]:
squad_requirements = { # General squad requirements
    "GK": 3,
    "DEF": 9,
    "MID": 7,
    "FWD": 7
}
squad_ratings = team_rating(players, requirements=squad_requirements, rating_name="squad_rating")

top11_requirements = { # 4-3-3 formation as estimate for starting 11 requirements
    "GK": 1,
    "DEF": 4,
    "MID": 3,
    "FWD": 3
}
top11_ratings = team_rating(players, requirements=top11_requirements, rating_name="top11_rating")

# Rename columns before merger
squad_ratings.rename(columns={"avg_rating": "squad_rating"}, inplace=True)
top11_ratings.rename(columns={"avg_rating": "top11_rating"}, inplace=True)

fixtures = add_ratings(fixtures, squad_ratings, top11_ratings)
fixtures

,date,home_team,away_team,home_score,away_score,tournament,city,host_country,neutral,home_squad_rating,away_squad_rating,home_top11_rating,away_top11_rating
0,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False,75.576923,59.384615,77.181818,66.818182
1,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True,73.576923,75.576923,76.090909,77.090909
2,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False,72.884615,72.807692,77.000000,76.272727
3,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False,76.423077,73.692308,79.090909,75.181818
4,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True,68.307692,77.615385,72.090909,80.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,52.346154,82.115385,55.545455,84.454545
68,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,77.346154,81.961538,79.454545,84.636364
69,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True,71.384615,53.730769,75.727273,58.818182
70,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,59.500000,83.461538,68.363636,85.727273


## Host Advantage

In [5]:
fixtures = add_host_advantage(fixtures)

The host_advantage feature is true if either the home or away team is one of the host nations (USA, Canada, Mexico). In later stages of the World Cup, the host nations may appear as away teams. Hence, away teams are also included in the definition.

## Elo features

In [6]:
# Checking for any discrepancies in country names between fixtures and elo ratings
fixtures_countries = set(fixtures["home_team"]).union(
    set(fixtures["away_team"])
)

elo_countries = set(elo_ratings["country_full"])

print("In fixtures but not elo:")
print(sorted(fixtures_countries - elo_countries))

print("\nIn elo but not fixtures:")
print(sorted(elo_countries - fixtures_countries))

In fixtures but not elo:
[]

In elo but not fixtures:
['Afghanistan', 'Albania', 'American Samoa', 'Andorra', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Armenia', 'Aruba', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bolivia', 'Botswana', 'British Virgin Islands', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cambodia', 'Cameroon', 'Cayman Islands', 'Central African Republic', 'Chad', 'Chile', 'China PR', 'Chinese Taipei', 'Comoros', 'Congo', 'Cook Islands', 'Costa Rica', 'Cuba', 'Cyprus', 'Czechoslovakia', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'Eswatini', 'Ethiopia', 'Faroe Islands', 'Fiji', 'Finland', 'Gabon', 'Georgia', 'Gibraltar', 'Greece', 'Grenada', 'Guam', 'Guatemala', 'Guinea', 'Guinea-Bissau', 'Guyana', 'Honduras', 'Hong Kong, China', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Israel', 'Italy', 'Jamaica', 'Kazakh

In [7]:
# Filtering for the latest elo ratings for countries in the World Cup 2026
elo_ratings_wc2026 = elo_ratings[elo_ratings["country_full"].isin(fixtures_countries)]
elo_ratings_wc2026 = elo_ratings_wc2026[elo_ratings_wc2026["rank_date"] == "2026-04-01"] # last update to elo ratings before the World Cup 2026
print(len(elo_ratings_wc2026)) # ensuring that all countries are included

48


In [8]:
fixtures = add_elo_features(fixtures, elo_ratings_wc2026)

fixtures

,date,home_team,away_team,home_score,away_score,tournament,city,host_country,neutral,home_squad_rating,away_squad_rating,home_top11_rating,away_top11_rating,host_advantage,elo_diff,abs_elo_diff
0,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False,75.576923,59.384615,77.181818,66.818182,1,334.0,334.0
1,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True,73.576923,75.576923,76.090909,77.090909,0,26.0,26.0
2,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False,72.884615,72.807692,77.000000,76.272727,1,190.0,190.0
3,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False,76.423077,73.692308,79.090909,75.181818,1,-112.0,112.0
4,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True,68.307692,77.615385,72.090909,80.000000,0,-464.0,464.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,52.346154,82.115385,55.545455,84.454545,0,-423.0,423.0
68,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,77.346154,81.961538,79.454545,84.636364,0,-9.0,9.0
69,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True,71.384615,53.730769,75.727273,58.818182,0,-72.0,72.0
70,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,59.500000,83.461538,68.363636,85.727273,0,-283.0,283.0


## Recent Form Features

In [9]:
# Creating a unified dataframe of the latest team features for both home and away teams
latest_team_features = pd.concat([
    matches[[
        "date",
        "home_team",
        "home_recent_win_rate",
        "home_recent_draw_rate",
        "home_avg_goals_last10",
        "home_avg_conceded_last10"
    ]].rename(columns={
        "home_team": "team",
        "home_recent_win_rate": "recent_win_rate",
        "home_recent_draw_rate": "recent_draw_rate",
        "home_avg_goals_last10": "avg_goals_last10",
        "home_avg_conceded_last10": "avg_conceded_last10"
    }),
    matches[[
        "date",
        "away_team",
        "away_recent_win_rate",
        "away_recent_draw_rate",
        "away_avg_goals_last10",
        "away_avg_conceded_last10"
    ]].rename(columns={
        "away_team": "team",
        "away_recent_win_rate": "recent_win_rate",
        "away_recent_draw_rate": "recent_draw_rate",
        "away_avg_goals_last10": "avg_goals_last10",
        "away_avg_conceded_last10": "avg_conceded_last10"
    })
])

# Extracting the latest features for each country
latest_team_features = (
    latest_team_features
    .sort_values("date")
    .groupby("team")
    .tail(1)
)

In [10]:
fixtures = add_recent_stats(fixtures, latest_team_features)
fixtures

,date,home_team,away_team,home_score,away_score,tournament,city,host_country,neutral,home_squad_rating,...,away_recent_win_rate,away_recent_draw_rate,away_avg_goals_last10,away_avg_conceded_last10,recent_form_diff,abs_recent_form_diff,recent_draw_diff,abs_recent_draw_diff,diff_in_avg_goals,diff_in_avg_conceded
0,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False,75.576923,...,0.4,0.3,1.5,1.1,0.0,0.0,0.1,0.1,-0.4,-0.3
1,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True,73.576923,...,0.4,0.4,1.8,1.2,0.2,0.2,-0.3,0.3,0.0,0.0
2,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False,72.884615,...,0.4,0.4,2.1,1.1,0.0,0.0,0.1,0.1,-1.0,-0.7
3,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False,76.423077,...,0.4,0.3,1.1,1.0,0.1,0.1,-0.2,0.2,0.6,0.6
4,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True,68.307692,...,0.6,0.3,2.5,0.8,-0.5,0.5,0.1,0.1,-1.6,1.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,52.346154,...,0.8,0.1,2.1,0.4,-0.3,0.3,0.1,0.1,-0.1,1.1
68,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,77.346154,...,0.6,0.3,2.9,1.2,-0.1,0.1,0.0,0.0,-0.8,-0.2
69,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True,71.384615,...,0.4,0.5,1.5,0.7,0.3,0.3,-0.3,0.3,-0.3,-0.4
70,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,59.500000,...,0.8,0.1,2.5,0.4,-0.4,0.4,0.3,0.3,-1.0,1.0


## Tournament Weight

In [11]:
fixtures = add_tournament_weight(fixtures)

In [12]:
fixtures.info()

<class 'pandas.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      72 non-null     str    
 1   home_team                 72 non-null     str    
 2   away_team                 72 non-null     str    
 3   home_score                0 non-null      float64
 4   away_score                0 non-null      float64
 5   tournament                72 non-null     str    
 6   city                      72 non-null     str    
 7   host_country              72 non-null     str    
 8   neutral                   72 non-null     bool   
 9   home_squad_rating         72 non-null     float64
 10  away_squad_rating         72 non-null     float64
 11  home_top11_rating         72 non-null     float64
 12  away_top11_rating         72 non-null     float64
 13  host_advantage            72 non-null     int64  
 14  elo_diff               

# Generate Match Probabilities

In [13]:
# Selecting the features for the model
feature_cols = [
    "neutral",
    "host_advantage",
    "elo_diff",
    "abs_elo_diff",
    "recent_form_diff",
    "abs_recent_form_diff",
    "home_recent_draw_rate",
    "away_recent_draw_rate",
    "recent_draw_diff",
    "abs_recent_draw_diff",
    "home_avg_goals_last10",
    "away_avg_goals_last10",
    "diff_in_avg_goals",
    "home_avg_conceded_last10",
    "away_avg_conceded_last10",
    "diff_in_avg_conceded",
    "tournament_weight"
]

X_wc = fixtures[feature_cols]

### Generating Group Stage Match Probabilities based on trained features

In [14]:
# Predicting probabilities for each match outcome using the trained model
probs = model.predict_proba(X_wc)

In [15]:
# Correlation between squad ratings and top11 ratings 
squad_ratings["squad_rating"].corr(
    top11_ratings["top11_rating"]
)

0.980724675966236

As the squad and top 11 player ratings are highly correlated, squad rating is ignored. Top 11 rating better reflects the players who are likely to decide the outcome of matches.

### Adjust probabilities based on Top 11 Player Ratings

In [16]:
def adjust_probs_with_squad(probs, home_rating, away_rating):
    """
    probs: array of [away_win, draw, home_win] probabilities from model
    """
    rating_diff = home_rating - away_rating
    
    # Normalise rating diff to a small adjustment factor
    adjustment = np.clip(rating_diff / 500, -0.1, 0.1)  # tune the divisor
    
    adjusted = probs.copy()
    adjusted[2] += adjustment    # home win probability
    adjusted[0] -= adjustment    # away win probability
    # draw stays the same
    
    # Clip negatives and renormalise so they sum to 1
    adjusted = np.clip(adjusted, 0.05, 1)  # minimum probability is 5%
    adjusted = adjusted / adjusted.sum()   # renormalise
    
    return adjusted

# Apply to fixtures
for i, row in fixtures.iterrows():
    base_probs = model.predict_proba(X_wc.iloc[[i]])[0]
    home_rating = row["home_top11_rating"]
    away_rating = row["away_top11_rating"]
    
    adjusted = adjust_probs_with_squad(base_probs, home_rating, away_rating)
    print(f"{row['home_team']} vs {row['away_team']}: {adjusted}")

Mexico vs South Africa: [0.16774892 0.32402096 0.50823012]
South Korea vs Czech Republic: [0.32916165 0.37673196 0.2941064 ]
Canada vs Bosnia and Herzegovina: [0.18818117 0.3441898  0.46762903]
United States vs Paraguay: [0.38317371 0.36558543 0.25124085]
Qatar vs Switzerland: [0.66248362 0.25276376 0.08475262]
Brazil vs Morocco: [0.26534589 0.36136794 0.37328618]
Haiti vs Scotland: [0.540531   0.32190205 0.13756695]
Australia vs Turkey: [0.44179429 0.36033561 0.1978701 ]
Germany vs Curaçao: [0.13901753 0.3029575  0.55802496]
Ivory Coast vs Ecuador: [0.41567766 0.33801691 0.24630543]
Netherlands vs Japan: [0.2775501  0.38773052 0.33471938]
Sweden vs Tunisia: [0.26919573 0.3841555  0.34664878]
Belgium vs Egypt: [0.19005009 0.35127332 0.45867659]
Iran vs New Zealand: [0.19781877 0.32568041 0.47650082]
Spain vs Cape Verde: [0.15214318 0.30033191 0.5475249 ]
Saudi Arabia vs Uruguay: [0.49683624 0.33421681 0.16894695]
France vs Senegal: [0.21533516 0.35022552 0.43443931]
Iraq vs Norway: [0.

### Storing Probabilities

In [17]:
fixtures["away_win_prob"] = np.nan
fixtures["draw_prob"] = np.nan
fixtures["home_win_prob"] = np.nan

for i, row in fixtures.iterrows():
    base_probs = model.predict_proba(X_wc.iloc[[i]])[0]

    adjusted = adjust_probs_with_squad(
        base_probs,
        row["home_top11_rating"],
        row["away_top11_rating"]
    )

    fixtures.loc[i, "away_win_prob"] = adjusted[0]
    fixtures.loc[i, "draw_prob"] = adjusted[1]
    fixtures.loc[i, "home_win_prob"] = adjusted[2]

In [18]:
fixtures

,date,home_team,away_team,home_score,away_score,tournament,city,host_country,neutral,home_squad_rating,...,recent_form_diff,abs_recent_form_diff,recent_draw_diff,abs_recent_draw_diff,diff_in_avg_goals,diff_in_avg_conceded,tournament_weight,away_win_prob,draw_prob,home_win_prob
0,2026-06-11,Mexico,South Africa,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False,75.576923,...,0.0,0.0,0.1,0.1,-0.4,-0.3,5,0.167749,0.324021,0.508230
1,2026-06-11,South Korea,Czech Republic,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True,73.576923,...,0.2,0.2,-0.3,0.3,0.0,0.0,5,0.329162,0.376732,0.294106
2,2026-06-12,Canada,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Toronto,Canada,False,72.884615,...,0.0,0.0,0.1,0.1,-1.0,-0.7,5,0.188181,0.344190,0.467629
3,2026-06-12,United States,Paraguay,NaN,NaN,FIFA World Cup,Inglewood,United States,False,76.423077,...,0.1,0.1,-0.2,0.2,0.6,0.6,5,0.383174,0.365585,0.251241
4,2026-06-13,Qatar,Switzerland,NaN,NaN,FIFA World Cup,Santa Clara,United States,True,68.307692,...,-0.5,0.5,0.1,0.1,-1.6,1.1,5,0.662484,0.252764,0.084753
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,52.346154,...,-0.3,0.3,0.1,0.1,-0.1,1.1,5,0.635620,0.296534,0.067846
68,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,77.346154,...,-0.1,0.1,0.0,0.0,-0.8,-0.2,5,0.371678,0.372979,0.255343
69,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True,71.384615,...,0.3,0.3,-0.3,0.3,-0.3,-0.4,5,0.361898,0.372081,0.266021
70,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True,59.500000,...,-0.4,0.4,0.3,0.3,-1.0,1.0,5,0.606225,0.297734,0.096041


# Simulation of One Tournament

My experimentation before creating simulation.py

## GROUP STAGE

### Simulate Match Outcomes

In [19]:
# Function to simulate match outcomes
def simulate_match(row):
    probs = [
        row["away_win_prob"],
        row["draw_prob"],
        row["home_win_prob"]
    ]

    return np.random.choice(
        ["away_win", "draw", "home_win"],
        p=probs
    )

# Simulate Group Stage matches
fixtures["sim_result"] = fixtures.apply(
    simulate_match,
    axis=1
)

### Simulate Match Scores

In [20]:
# Function to simulate match scores
def simulate_score(result, home_attack, away_attack):

    while True:

        home_goals = np.random.poisson(home_attack)
        away_goals = np.random.poisson(away_attack)

        if result == "home_win" and home_goals > away_goals:
            return home_goals, away_goals

        elif result == "away_win" and away_goals > home_goals:
            return home_goals, away_goals

        elif result == "draw" and home_goals == away_goals:
            return home_goals, away_goals

# Simulate match scores of Group Stage matches
fixtures[["sim_home_goals", "sim_away_goals"]] = (
    fixtures.apply(
        lambda row: pd.Series(
            simulate_score(
                row["sim_result"],
                row["home_avg_goals_last10"],
                row["away_avg_goals_last10"]
            )
        ),
        axis=1
    )
)

### Build Standings Dataframe

In [21]:
# Create list of countries
teams = wc2026_draw["country"].unique()

# Initialise standings dataframe
standings = pd.DataFrame({
    "country": teams,
    "points": 0,
    "gf": 0,
    "ga": 0
})

standings.head()

,country,points,gf,ga
0,Mexico,0,0,0
1,South Africa,0,0,0
2,South Korea,0,0,0
3,Czech Republic,0,0,0
4,Canada,0,0,0


### Update Standings Based on Simulated Match Results

In [22]:
for _, row in fixtures.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    hg = row["sim_home_goals"]
    ag = row["sim_away_goals"]

    # Goals for / against
    standings.loc[
        standings["country"] == home,
        ["gf", "ga"]
    ] += [hg, ag]

    standings.loc[
        standings["country"] == away,
        ["gf", "ga"]
    ] += [ag, hg]

    # Points
    if hg > ag:
        standings.loc[
            standings["country"] == home,
            "points"
        ] += 3

    elif ag > hg:
        standings.loc[
            standings["country"] == away,
            "points"
        ] += 3

    else:
        standings.loc[
            standings["country"] == home,
            "points"
        ] += 1

        standings.loc[
            standings["country"] == away,
            "points"
        ] += 1


# Create goal difference column
standings["gd"] = standings["gf"] - standings["ga"]

# Add Groups
standings = standings.merge(
    wc2026_draw[["country", "group"]],
    on="country",
    how="left"
)

# Sort by group, points, goal difference, goals for
standings = standings.sort_values(
    ["group", "points", "gd", "gf"],
    ascending=[True, False, False, False]
)

# Rearranging columns
standings = standings[["group", "country", "points", "gf", "ga", "gd"]]

# Create position column within each group
standings["position"] = (
    standings.groupby("group")
    .cumcount()
    + 1
)
standings

,group,country,points,gf,ga,gd,position
3,A,Czech Republic,7,7,1,6,1
0,A,Mexico,4,3,4,-1,2
2,A,South Korea,3,2,6,-4,3
1,A,South Africa,2,4,5,-1,4
7,B,Switzerland,6,3,4,-1,1
4,B,Canada,5,4,1,3,2
5,B,Bosnia and Herzegovina,2,3,4,-1,3
6,B,Qatar,2,1,2,-1,4
8,C,Brazil,5,7,6,1,1
9,C,Morocco,3,7,7,0,2


### Identifying teams that qualify to the Round of 32

In [23]:
# List of countries in WC 2026
wc2026_countries = wc2026_draw["country"].tolist()

# Initialise Dictionary of FIFA rankings
fifa_rank = {country: 0 for country in wc2026_countries}
# Update with the current FIFA rankings as of 5 June 2026
fifa_rank.update({
    "Argentina": 1,
    "Spain": 2,
    "France": 3,
    "England": 4,
    "Portugal": 5,
    "Brazil": 6,
    "Morocco": 7,
    "Netherlands": 8,
    "Belgium": 9,
    "Germany": 10,
    "Croatia": 11,
    "Colombia": 13,
    "Mexico": 14,
    "Senegal": 15,
    "United States": 16,
    "Uruguay": 17,
    "Japan": 18,
    "Switzerland": 19,
    "Iran": 20,
    "Turkey": 22,
    "Austria": 23,
    "Ecuador": 24,
    "South Korea": 25,
    "Australia": 27,
    "Algeria": 28,
    "Egypt": 29,
    "Canada": 30,
    "Norway": 31,
    "Ivory Coast": 33,
    "Panama": 34,
    "Sweden": 38,
    "Czech Republic": 39,
    "Paraguay": 40,
    "Scotland": 43,
    "DR Congo": 45,
    "Tunisia": 46,
    "Uzbekistan": 50,
    "Qatar": 55,
    "Iraq": 56,
    "South Africa": 60,
    "Saudi Arabia": 61,
    "Jordan": 63,
    "Bosnia and Herzegovina": 64,
    "Cape Verde": 68,
    "Ghana": 73,
    "Haiti": 81,
    "Curaçao": 83,
    "New Zealand": 85
})

# Checking to ensure every country has a non-zero rank
zero_rank_teams = [team for team, rank in fifa_rank.items() if rank == 0]
print(zero_rank_teams) 

[]


In [24]:
standings["fifa_rank"] = standings["country"].map(fifa_rank)

# Identify 1st, 2nd and 3rd place teams in each group that qualify to the Round of 32
group_winners = standings[
    standings["position"] == 1
]

group_runners_up = standings[
    standings["position"] == 2
]

third_place = standings[
    standings["position"] == 3
]

best_third = (
    third_place
    .sort_values(
        ["points", "gd", "gf", "fifa_rank"],
        ascending=[False, False, False, True]
    )
    .head(8)
)

# Identify which groups the best third place teams are from
best_third_groups = set(best_third["group"].values)

According to FIFA's rules, the best third-place teams are determined by:
1. Points
2. Goal Difference
3. Goals Scored
4. Team Conduct Score
5. FIFA World Ranking

where Team Conduct Score is calculated by the number of yellow and red cards accumulated. For simplicity, Team Conduct Score is omitted.

## ROUND OF 32

### Combinations of matches in the Round of 32

In [25]:
group_cols = list("ABCDEFGHIJKL")

# Create combo column in mapping
def make_combo(row):
    return "".join(
        g
        for g in group_cols
        if pd.notna(row[g])
    )

mapping["combo"] = mapping.apply(make_combo, axis=1)

In [26]:
# Check to see how the mapping df looks like
mapping

,A,B,C,D,E,F,G,H,I,J,...,L,1A,1B,1D,1E,1G,1I,1K,1L,combo
0,NaN,NaN,NaN,NaN,E,F,G,H,I,J,...,L,3E,3J,3I,3F,3H,3G,3L,3K,EFGHIJKL
1,NaN,NaN,NaN,D,NaN,F,G,H,I,J,...,L,3H,3G,3I,3D,3J,3F,3L,3K,DFGHIJKL
2,NaN,NaN,NaN,D,E,NaN,G,H,I,J,...,L,3E,3J,3I,3D,3H,3G,3L,3K,DEGHIJKL
3,NaN,NaN,NaN,D,E,F,NaN,H,I,J,...,L,3E,3J,3I,3D,3H,3F,3L,3K,DEFHIJKL
4,NaN,NaN,NaN,D,E,F,G,NaN,I,J,...,L,3E,3G,3I,3D,3J,3F,3L,3K,DEFGIJKL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
490,A,B,C,D,E,F,G,NaN,NaN,NaN,...,L,3C,3G,3B,3D,3A,3F,3L,3E,ABCDEFGL
491,A,B,C,D,E,F,G,NaN,NaN,NaN,...,NaN,3C,3G,3B,3D,3A,3F,3E,3K,ABCDEFGK
492,A,B,C,D,E,F,G,NaN,NaN,J,...,NaN,3C,3G,3B,3D,3A,3F,3E,3J,ABCDEFGJ
493,A,B,C,D,E,F,G,NaN,I,NaN,...,NaN,3C,3G,3B,3D,3A,3F,3E,3I,ABCDEFGI


In [27]:
# Get the combination of the 8 best third place groups from the simulation
best_third_groups = sorted(best_third["group"].values)
combo_key = "".join(best_third_groups)
print(f"Combo key: {combo_key}")

# Look up the right row
matchup_row = mapping[mapping["combo"] == combo_key].iloc[0]
print(matchup_row)

Combo key: CFGHIJKL
A             NaN
B             NaN
C               C
D             NaN
E             NaN
F               F
G               G
H               H
I               I
J               J
K               K
L               L
1A             3H
1B             3G
1D             3I
1E             3C
1G             3J
1I             3F
1K             3L
1L             3K
combo    CFGHIJKL
Name: 9, dtype: str


In [28]:
standings["position_group"] = standings["position"].astype(str) + standings["group"]
standings

,group,country,points,gf,ga,gd,position,fifa_rank,position_group
3,A,Czech Republic,7,7,1,6,1,39,1A
0,A,Mexico,4,3,4,-1,2,14,2A
2,A,South Korea,3,2,6,-4,3,25,3A
1,A,South Africa,2,4,5,-1,4,60,4A
7,B,Switzerland,6,3,4,-1,1,19,1B
4,B,Canada,5,4,1,3,2,30,2B
5,B,Bosnia and Herzegovina,2,3,4,-1,3,64,3B
6,B,Qatar,2,1,2,-1,4,55,4B
8,C,Brazil,5,7,6,1,1,6,1C
9,C,Morocco,3,7,7,0,2,7,2C


### Preparation

In [29]:
# Create a df of match brackets as per the rules derived from the official FIFA website
bracket = pd.DataFrame([
    # Round of 32
    [73, "R32", "2A", "2B", "Los Angeles Stadium"],
    [74, "R32", "1E", "THIRD_E", "Boston Stadium"],
    [75, "R32", "1F", "2C", "Estadio Monterrey"],
    [76, "R32", "1C", "2F", "Houston Stadium"],
    [77, "R32", "1I", "THIRD_I", "New York New Jersey Stadium"],
    [78, "R32", "2E", "2I", "Dallas Stadium"],
    [79, "R32", "1A", "THIRD_A", "Mexico City Stadium"],
    [80, "R32", "1L", "THIRD_L", "Atlanta Stadium"],
    [81, "R32", "1D", "THIRD_D", "San Francisco Bay Area Stadium"],
    [82, "R32", "1G", "THIRD_G", "Seattle Stadium"],
    [83, "R32", "2K", "2L", "Toronto Stadium"],
    [84, "R32", "1H", "2J", "Los Angeles Stadium"],
    [85, "R32", "1B", "THIRD_B", "BC Place Vancouver"],
    [86, "R32", "1J", "2H", "Miami Stadium"],
    [87, "R32", "1K", "THIRD_K", "Kansas City Stadium"],
    [88, "R32", "2D", "2G", "Dallas Stadium"],

    # Round of 16
    [89, "R16", "W74", "W77", "Philadelphia Stadium"],
    [90, "R16", "W73", "W75", "Houston Stadium"],
    [91, "R16", "W76", "W78", "New York New Jersey Stadium"],
    [92, "R16", "W79", "W80", "Mexico City Stadium"],
    [93, "R16", "W83", "W84", "Dallas Stadium"],
    [94, "R16", "W81", "W82", "Seattle Stadium"],
    [95, "R16", "W86", "W88", "Atlanta Stadium"],
    [96, "R16", "W85", "W87", "BC Place Vancouver"],

    # Quarter-finals
    [97, "QF", "W89", "W90", "Boston Stadium"],
    [98, "QF", "W93", "W94", "Los Angeles Stadium"],
    [99, "QF", "W91", "W92", "Miami Stadium"],
    [100, "QF", "W95", "W96", "Kansas City Stadium"],

    # Semi-finals
    [101, "SF", "W97", "W98", "Dallas Stadium"],
    [102, "SF", "W99", "W100", "Atlanta Stadium"],

    # Third-place playoff
    [103, "3P", "L101", "L102", "Miami Stadium"],

    # Final
    [104, "F", "W101", "W102", "New York New Jersey Stadium"],
], columns=[
    "match_id",
    "round",
    "team1_source",
    "team2_source",
    "stadium"
])

third_mapping = {
    "THIRD_A": matchup_row["1A"],
    "THIRD_B": matchup_row["1B"],
    "THIRD_D": matchup_row["1D"],
    "THIRD_E": matchup_row["1E"],
    "THIRD_G": matchup_row["1G"],
    "THIRD_I": matchup_row["1I"],
    "THIRD_K": matchup_row["1K"],
    "THIRD_L": matchup_row["1L"],
}

In [30]:
# Initialise stadium mapping dictionary
stadium_list = bracket["stadium"].tolist()
stadium_country = {country: "" for country in stadium_list}

stadium_country.update({
    'Los Angeles Stadium': 'United States',
    'Boston Stadium': 'Unites States',
    'Estadio Monterrey': 'Mexico',
    'Houston Stadium': 'United States',
    'New York New Jersey Stadium': 'United States',
    'Dallas Stadium': 'United States',
    'Mexico City Stadium': 'Mexico',
    'Atlanta Stadium': 'United States',
    'San Francisco Bay Area Stadium': 'United States',
    'Seattle Stadium': 'United States',
    'Toronto Stadium': 'Canada',
    'BC Place Vancouver': 'Canada',
    'Miami Stadium': 'United States',
    'Kansas City Stadium': 'United States',
    'Philadelphia Stadium': 'United States'
})

stadium_country

{'Los Angeles Stadium': 'United States',
 'Boston Stadium': 'Unites States',
 'Estadio Monterrey': 'Mexico',
 'Houston Stadium': 'United States',
 'New York New Jersey Stadium': 'United States',
 'Dallas Stadium': 'United States',
 'Mexico City Stadium': 'Mexico',
 'Atlanta Stadium': 'United States',
 'San Francisco Bay Area Stadium': 'United States',
 'Seattle Stadium': 'United States',
 'Toronto Stadium': 'Canada',
 'BC Place Vancouver': 'Canada',
 'Miami Stadium': 'United States',
 'Kansas City Stadium': 'United States',
 'Philadelphia Stadium': 'United States'}

In [31]:
bracket["host_country"] = bracket["stadium"].map(stadium_country)
bracket

,match_id,round,team1_source,team2_source,stadium,host_country
0,73,R32,2A,2B,Los Angeles Stadium,United States
1,74,R32,1E,THIRD_E,Boston Stadium,Unites States
2,75,R32,1F,2C,Estadio Monterrey,Mexico
3,76,R32,1C,2F,Houston Stadium,United States
4,77,R32,1I,THIRD_I,New York New Jersey Stadium,United States
5,78,R32,2E,2I,Dallas Stadium,United States
6,79,R32,1A,THIRD_A,Mexico City Stadium,Mexico
7,80,R32,1L,THIRD_L,Atlanta Stadium,United States
8,81,R32,1D,THIRD_D,San Francisco Bay Area Stadium,United States
9,82,R32,1G,THIRD_G,Seattle Stadium,United States


In [32]:
# Create dictionaries for the first place, second place, third place of each group
first_place_dict = (
    standings[standings["position"] == 1]
    .set_index("group")["country"]
    .to_dict()
)

second_place_dict = (
    standings[standings["position"] == 2]
    .set_index("group")["country"]
    .to_dict()
)

third_place_dict = (
    standings[standings["position"] == 3]
    .set_index("group")["country"]
    .to_dict()
)

In [33]:
# Function to look for the teams playing against each other in the Round of 32
def resolve_r32(source):

    # Group winners
    if source.startswith("1"):
        return first_place_dict[source[1]]

    # Group runners-up
    elif source.startswith("2"):
        return second_place_dict[source[1]]

    # Third-place placeholders
    elif source.startswith("THIRD"):

        third_slot = third_mapping[source]
        group = third_slot[1]

        return third_place_dict[group]

    else:
        raise ValueError(f"Cannot resolve {source}")

In [34]:
r32 = bracket[bracket["round"] == "R32"].copy()

r32["home_team"] = r32["team1_source"].apply(resolve_r32)
r32["away_team"] = r32["team2_source"].apply(resolve_r32)

r32

,match_id,round,team1_source,team2_source,stadium,host_country,home_team,away_team
0,73,R32,2A,2B,Los Angeles Stadium,United States,Mexico,Canada
1,74,R32,1E,THIRD_E,Boston Stadium,Unites States,Germany,Scotland
2,75,R32,1F,2C,Estadio Monterrey,Mexico,Japan,Morocco
3,76,R32,1C,2F,Houston Stadium,United States,Brazil,Netherlands
4,77,R32,1I,THIRD_I,New York New Jersey Stadium,United States,Norway,Tunisia
5,78,R32,2E,2I,Dallas Stadium,United States,Curaçao,Senegal
6,79,R32,1A,THIRD_A,Mexico City Stadium,Mexico,Czech Republic,Saudi Arabia
7,80,R32,1L,THIRD_L,Atlanta Stadium,United States,England,Uzbekistan
8,81,R32,1D,THIRD_D,San Francisco Bay Area Stadium,United States,Australia,Iraq
9,82,R32,1G,THIRD_G,Seattle Stadium,United States,Iran,Algeria


#### Full Round of 32 Dataset

In [35]:
r32 = create_match_features(r32, squad_ratings, top11_ratings, elo_ratings_wc2026, latest_team_features)
r32["tournament_weight"] = 5 # World Cup Tournament is defined to have a tournament weight of 5
r32

,match_id,round,team1_source,team2_source,stadium,host_country,home_team,away_team,home_squad_rating,away_squad_rating,...,away_recent_draw_rate,away_avg_goals_last10,away_avg_conceded_last10,recent_form_diff,abs_recent_form_diff,recent_draw_diff,abs_recent_draw_diff,diff_in_avg_goals,diff_in_avg_conceded,tournament_weight
0,73,R32,2A,2B,Los Angeles Stadium,United States,Mexico,Canada,75.576923,72.884615,...,0.5,1.1,0.4,0.0,0.0,-0.1,0.1,0.0,0.4,5
1,74,R32,1E,THIRD_E,Boston Stadium,Unites States,Germany,Scotland,83.153846,75.076923,...,0.1,1.8,1.2,0.2,0.2,-0.1,0.1,0.5,-0.1,5
2,75,R32,1F,2C,Estadio Monterrey,Mexico,Japan,Morocco,75.346154,77.807692,...,0.3,2.0,0.3,-0.1,0.1,-0.1,0.1,-0.2,0.4,5
3,76,R32,1C,2F,Houston Stadium,United States,Brazil,Netherlands,83.615385,81.730769,...,0.3,3.0,0.6,-0.2,0.2,-0.1,0.1,-1.2,0.2,5
4,77,R32,1I,THIRD_I,New York New Jersey Stadium,United States,Norway,Tunisia,76.423077,65.115385,...,0.5,1.4,1.0,0.4,0.4,-0.3,0.3,1.7,-0.4,5
5,78,R32,2E,2I,Dallas Stadium,United States,Curaçao,Senegal,58.384615,76.115385,...,0.1,2.5,0.6,-0.5,0.5,0.3,0.3,-0.9,0.7,5
6,79,R32,1A,THIRD_A,Mexico City Stadium,Mexico,Czech Republic,Saudi Arabia,75.576923,69.807692,...,0.1,0.9,1.3,0.0,0.0,0.3,0.3,0.9,-0.1,5
7,80,R32,1L,THIRD_L,Atlanta Stadium,United States,England,Uzbekistan,83.461538,53.730769,...,0.5,1.5,0.7,0.4,0.4,-0.4,0.4,1.0,-0.3,5
8,81,R32,1D,THIRD_D,San Francisco Bay Area Stadium,United States,Australia,Iraq,70.846154,57.423077,...,0.2,1.1,0.7,0.1,0.1,-0.2,0.2,0.4,0.2,5
9,82,R32,1G,THIRD_G,Seattle Stadium,United States,Iran,Algeria,54.846154,72.384615,...,0.2,2.3,0.5,-0.3,0.3,0.1,0.1,-0.7,0.2,5


In [36]:
# Checking to ensure that all necessary features are included
set(feature_cols) - set(r32.columns)

set()

### Simulation of the Round of 32

In [37]:
# Function to simulate matches in the knockout stages (Round of 32, Round of 16, Quarter-Finals, Semi-Final, Final)
def simulate_knockout_match(match_row):

    X = match_row[feature_cols].to_frame().T

    probs = model.predict_proba(X)[0]

    adjusted = adjust_probs_with_squad(
        probs,
        match_row["home_top11_rating"],
        match_row["away_top11_rating"]
    )

    result = np.random.choice(
        ["away_win", "draw", "home_win"],
        p=adjusted
    )

    if result == "home_win":
        return match_row["home_team"]

    elif result == "away_win":
        return match_row["away_team"]

    else:
        # penalties
        return np.random.choice([
            match_row["home_team"],
            match_row["away_team"]
        ])

In [38]:
# Simulating and saving the winners in a dictionary
r32_winners = {}

for _, match in r32.iterrows():

    winner = simulate_knockout_match(match)

    r32_winners[match["match_id"]] = winner

In [39]:
r32_winners

{73: 'Mexico',
 74: 'Germany',
 75: 'Japan',
 76: 'Netherlands',
 77: 'Norway',
 78: 'Senegal',
 79: 'Czech Republic',
 80: 'England',
 81: 'Iraq',
 82: 'Iran',
 83: 'Croatia',
 84: 'Spain',
 85: 'Switzerland',
 86: 'Cape Verde',
 87: 'Panama',
 88: 'Belgium'}

## ROUND OF 16

### Preparation

In [40]:
r16 = bracket[bracket["round"] == "R16"].copy()
r16

,match_id,round,team1_source,team2_source,stadium,host_country
16,89,R16,W74,W77,Philadelphia Stadium,United States
17,90,R16,W73,W75,Houston Stadium,United States
18,91,R16,W76,W78,New York New Jersey Stadium,United States
19,92,R16,W79,W80,Mexico City Stadium,Mexico
20,93,R16,W83,W84,Dallas Stadium,United States
21,94,R16,W81,W82,Seattle Stadium,United States
22,95,R16,W86,W88,Atlanta Stadium,United States
23,96,R16,W85,W87,BC Place Vancouver,Canada


In [41]:
# Function to find the respective teams for matches
def resolve_team(source, team):
    if source.startswith("W"): 
        match_id = int(source[1:]) # Match ID number
        return team[match_id]
    elif source.startswith("L"):
        match_id = int(source[1:])
        return team[match_id]

    raise ValueError(f"Unexpected source: {source}")

r16["home_team"] = r16["team1_source"].apply(lambda x: resolve_team(x, r32_winners))
r16["away_team"] = r16["team2_source"].apply(lambda x: resolve_team(x, r32_winners))

#### Full Round of 16 Dataset

In [42]:
r16 = create_match_features(r16, squad_ratings, top11_ratings, elo_ratings_wc2026, latest_team_features)
r16["tournament_weight"] = 5
r16

,match_id,round,team1_source,team2_source,stadium,host_country,home_team,away_team,home_squad_rating,away_squad_rating,...,away_recent_draw_rate,away_avg_goals_last10,away_avg_conceded_last10,recent_form_diff,abs_recent_form_diff,recent_draw_diff,abs_recent_draw_diff,diff_in_avg_goals,diff_in_avg_conceded,tournament_weight
0,89,R16,W74,W77,Philadelphia Stadium,United States,Germany,Norway,83.153846,76.423077,...,0.2,3.1,0.6,0.0,0.0,-0.2,0.2,-0.8,0.5,5
1,90,R16,W73,W75,Houston Stadium,United States,Mexico,Japan,75.576923,75.346154,...,0.2,1.8,0.7,-0.2,0.2,0.2,0.2,-0.7,0.1,5
2,91,R16,W76,W78,New York New Jersey Stadium,United States,Netherlands,Senegal,81.730769,76.115385,...,0.1,2.5,0.6,-0.1,0.1,0.2,0.2,0.5,0.0,5
3,92,R16,W79,W80,Mexico City Stadium,Mexico,Czech Republic,England,75.576923,83.461538,...,0.1,2.5,0.4,-0.4,0.4,0.3,0.3,-0.7,0.8,5
4,93,R16,W83,W84,Dallas Stadium,United States,Croatia,Spain,77.807692,84.230769,...,0.3,3.4,1.1,0.1,0.1,-0.2,0.2,-0.5,-0.3,5
5,94,R16,W81,W82,Seattle Stadium,United States,Iraq,Iran,57.423077,54.846154,...,0.3,1.6,0.7,0.2,0.2,-0.1,0.1,-0.5,0.0,5
6,95,R16,W86,W88,Atlanta Stadium,United States,Cape Verde,Belgium,65.307692,80.115385,...,0.4,3.5,1.0,-0.2,0.2,0.1,0.1,-1.8,0.0,5
7,96,R16,W85,W87,BC Place Vancouver,Canada,Switzerland,Panama,77.615385,59.500000,...,0.4,1.5,1.4,0.2,0.2,-0.1,0.1,1.0,-0.6,5


### Simulation of the Round of 16

In [43]:
r16_winners = {}

for _, match in r16.iterrows():

    winner = simulate_knockout_match(match)

    r16_winners[match["match_id"]] = winner

In [44]:
r16_winners

{89: 'Norway',
 90: 'Japan',
 91: 'Netherlands',
 92: 'Czech Republic',
 93: 'Spain',
 94: 'Iran',
 95: 'Cape Verde',
 96: 'Switzerland'}

## QUARTER-FINALS

### Preparation

In [45]:
qf = bracket[bracket["round"]=="QF"].copy()

qf["home_team"] = qf["team1_source"].apply(lambda x: resolve_team(x, r16_winners))
qf["away_team"] = qf["team2_source"].apply(lambda x: resolve_team(x, r16_winners))

#### Full QF Dataset

In [46]:
qf = create_match_features(qf, squad_ratings, top11_ratings, elo_ratings_wc2026, latest_team_features)
qf["tournament_weight"] = 5
qf

,match_id,round,team1_source,team2_source,stadium,host_country,home_team,away_team,home_squad_rating,away_squad_rating,...,away_recent_draw_rate,away_avg_goals_last10,away_avg_conceded_last10,recent_form_diff,abs_recent_form_diff,recent_draw_diff,abs_recent_draw_diff,diff_in_avg_goals,diff_in_avg_conceded,tournament_weight
0,97,QF,W89,W90,Boston Stadium,Unites States,Norway,Japan,76.423077,75.346154,...,0.2,1.8,0.7,0.1,0.1,0.0,0.0,1.3,-0.1,5
1,98,QF,W93,W94,Los Angeles Stadium,United States,Spain,Iran,84.230769,54.846154,...,0.3,1.6,0.7,0.3,0.3,0.0,0.0,1.8,0.4,5
2,99,QF,W91,W92,Miami Stadium,United States,Netherlands,Czech Republic,81.730769,75.576923,...,0.4,1.8,1.2,0.3,0.3,-0.1,0.1,1.2,-0.6,5
3,100,QF,W95,W96,Kansas City Stadium,United States,Cape Verde,Switzerland,65.307692,77.615385,...,0.3,2.5,0.8,-0.2,0.2,0.2,0.2,-0.8,0.2,5


### Simulation of the Quarter-Finals

In [47]:
qf_winners = {}

for _, match in qf.iterrows():

    winner = simulate_knockout_match(match)

    qf_winners[match["match_id"]] = winner

In [48]:
qf_winners

{97: 'Japan', 98: 'Spain', 99: 'Netherlands', 100: 'Cape Verde'}

## SEMI-FINALS

### Preparation

In [49]:
sf = bracket[bracket["round"]=="SF"].copy()

sf["home_team"] = sf["team1_source"].apply(lambda x: resolve_team(x, qf_winners))
sf["away_team"] = sf["team2_source"].apply(lambda x: resolve_team(x, qf_winners))

#### Full Semi-Finals Dataset

In [50]:
sf = create_match_features(sf, squad_ratings, top11_ratings, elo_ratings_wc2026, latest_team_features)
sf["tournament_weight"] = 5
sf

,match_id,round,team1_source,team2_source,stadium,host_country,home_team,away_team,home_squad_rating,away_squad_rating,...,away_recent_draw_rate,away_avg_goals_last10,away_avg_conceded_last10,recent_form_diff,abs_recent_form_diff,recent_draw_diff,abs_recent_draw_diff,diff_in_avg_goals,diff_in_avg_conceded,tournament_weight
0,101,SF,W97,W98,Dallas Stadium,United States,Japan,Spain,75.346154,84.230769,...,0.3,3.4,1.1,-0.1,0.1,-0.1,0.1,-1.6,-0.4,5
1,102,SF,W99,W100,Atlanta Stadium,United States,Netherlands,Cape Verde,81.730769,65.307692,...,0.5,1.7,1.0,0.3,0.3,-0.2,0.2,1.3,-0.4,5


### Simulation of the Semi-Finals

In [51]:
sf_winners = {}
sf_losers = {}

for _, match in sf.iterrows():

    winner = simulate_knockout_match(match)

    if winner == match["home_team"]:
        loser = match["away_team"]
    else:
        loser = match["home_team"]

    sf_winners[match["match_id"]] = winner
    sf_losers[match["match_id"]] = loser

In [52]:
print(sf_winners)
print(sf_losers)

{101: 'Japan', 102: 'Netherlands'}
{101: 'Spain', 102: 'Cape Verde'}


## THIRD-PLACE PLAYOFF

### Preparation

In [53]:
# Finding the teams playing in the third-place playoff
third = bracket[bracket["round"]=="3P"].copy()
third["home_team"] = third["team1_source"].apply(lambda x: resolve_team(x, sf_losers))
third["away_team"] = third["team2_source"].apply(lambda x: resolve_team(x, sf_losers))

#### Full Third-place Playoff Dataset

In [54]:
third = create_match_features(third, squad_ratings, top11_ratings, elo_ratings_wc2026, latest_team_features)
third["tournament_weight"] = 5
third

,match_id,round,team1_source,team2_source,stadium,host_country,home_team,away_team,home_squad_rating,away_squad_rating,...,away_recent_draw_rate,away_avg_goals_last10,away_avg_conceded_last10,recent_form_diff,abs_recent_form_diff,recent_draw_diff,abs_recent_draw_diff,diff_in_avg_goals,diff_in_avg_conceded,tournament_weight
0,103,3P,L101,L102,Miami Stadium,United States,Spain,Cape Verde,84.230769,65.307692,...,0.5,1.7,1.0,0.3,0.3,-0.2,0.2,1.7,0.1,5


### Simulation of the Third-place Playoff

In [55]:
for _, match in third.iterrows():
    third_place = simulate_knockout_match(match)

In [56]:
third_place

'Spain'

## FINAL

### Preparation

In [57]:
# Finding the respective teams playing in the Finals
final = bracket[bracket["round"]=="F"].copy()
final["home_team"] = final["team1_source"].apply(lambda x: resolve_team(x, sf_winners))
final["away_team"] = final["team2_source"].apply(lambda x: resolve_team(x, sf_winners))

#### Full Final Dataset

In [58]:
final = create_match_features(final, squad_ratings, top11_ratings, elo_ratings_wc2026, latest_team_features)
final["tournament_weight"] = 5
final

,match_id,round,team1_source,team2_source,stadium,host_country,home_team,away_team,home_squad_rating,away_squad_rating,...,away_recent_draw_rate,away_avg_goals_last10,away_avg_conceded_last10,recent_form_diff,abs_recent_form_diff,recent_draw_diff,abs_recent_draw_diff,diff_in_avg_goals,diff_in_avg_conceded,tournament_weight
0,104,F,W101,W102,New York New Jersey Stadium,United States,Japan,Netherlands,75.346154,81.730769,...,0.3,3.0,0.6,-0.1,0.1,-0.1,0.1,-1.2,0.1,5


### Simulation of the Final

In [59]:
for _, match in final.iterrows():
    champion = simulate_knockout_match(match)
    if champion == match["home_team"]:
        runner_up = match["away_team"]
    else:
        runner_up = match["home_team"]

champion

'Japan'

## Consolidated Simulation

In [60]:
from src.simulation import simulate_one_tournament, run_monte_carlo

one_sim = simulate_one_tournament(
    fixtures=fixtures,
    bracket=bracket,
    wc2026_draw=wc2026_draw,
    mapping=mapping,
    fifa_rank=fifa_rank,
    model=model,
    feature_cols=feature_cols,
    adjust_probs_with_squad=adjust_probs_with_squad,
    squad_ratings=squad_ratings,
    top11_ratings=top11_ratings,
    elo_ratings_wc2026=elo_ratings_wc2026,
    latest_team_features=latest_team_features,
    rng=np.random.default_rng(42),
)

one_sim["champion"], one_sim["runner_up"]

('Spain', 'Croatia')

# Monte Carlo Simulations

In [61]:
mc_results = run_monte_carlo(
    n_simulations=1000,
    fixtures=fixtures,
    bracket=bracket,
    wc2026_draw=wc2026_draw,
    mapping=mapping,
    fifa_rank=fifa_rank,
    model=model,
    feature_cols=feature_cols,
    adjust_probs_with_squad=adjust_probs_with_squad,
    squad_ratings=squad_ratings,
    top11_ratings=top11_ratings,
    elo_ratings_wc2026=elo_ratings_wc2026,
    latest_team_features=latest_team_features,
    random_seed=42,
)

probabilities = mc_results["probabilities"]
probabilities.head(20)

,country,r32_count,r32_prob,quarter_final_count,quarter_final_prob,semi_final_count,semi_final_prob,final_count,final_prob,champion_count,champion_prob
0,Spain,925,0.925,399,0.399,264,0.264,176,0.176,107,0.107
1,Argentina,882,0.882,356,0.356,219,0.219,148,0.148,92,0.092
2,France,866,0.866,388,0.388,235,0.235,136,0.136,87,0.087
3,England,916,0.916,353,0.353,205,0.205,127,0.127,69,0.069
4,Netherlands,910,0.910,349,0.349,179,0.179,98,0.098,53,0.053
5,Switzerland,916,0.916,295,0.295,158,0.158,89,0.089,46,0.046
6,Portugal,844,0.844,287,0.287,168,0.168,79,0.079,45,0.045
7,Belgium,904,0.904,306,0.306,153,0.153,79,0.079,40,0.040
8,Brazil,861,0.861,272,0.272,165,0.165,79,0.079,36,0.036
9,Colombia,846,0.846,217,0.217,113,0.113,58,0.058,33,0.033


In [64]:
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

BASE_DIR = Path.cwd().parent

probabilities.to_csv(
    BASE_DIR / "data" / "results" / f"wc2026_predictions_1000_{timestamp}.csv",
    index=False
)